In [1]:
import sys
sys.path.append('/app')
# unit_test/test_single_receptor.py
import torch
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np
import os

from src.environment import *
from src.physics import *
from objectives.loss import ProxyInformationLoss,ExactInformationLoss
from src.geometry import generate_receptor_indices
from src.plot_helper import *


In [4]:
CONF = {
        "n_units": 10,
        "n_families": 100,
        "N":2,
        "k_sub": 5,
        "batch_size": 1024,
        "epochs": 600,
        "lr": 0.05,
        "cov_weight":10.,
        "k":5
    }

# 1. SETUP
# -----------------------------------------------------
# We use 1 Unit, 1 Family. The receptor is a homopentamer (Unit 0 five times).

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

# Initialize Modules
conc_strategy = LogNormalConcentration(n_families=CONF['n_families'], init_mean=5.0)
env = LigandEnvironment(CONF['n_units'], CONF['n_families'], conc_model=conc_strategy).to(device)

physics = Receptor(CONF["n_units"], CONF["k_sub"]).to(device)
loss_fn = ProxyInformationLoss(cov_weight=CONF['cov_weight']) # Default bandwidth

# Create the receptor identity: [[0, 0, 0, 0, 0]]
#receptor_indices = torch.zeros(1, CONF["k_sub"], dtype=torch.long, device=device)
receptor_indices = generate_receptor_indices(n_units=CONF['n_units'],k_sub= CONF['k_sub'],n_sensors=CONF['N'])
#receptor_indices = torch.tensor([[0,0,0,0,0],[1,1,1,1,1]],dtype=torch.long)
print(receptor_indices)

# Optimizer
optimizer = optim.Adam(list(env.parameters()) + list(physics.parameters()), lr=CONF["lr"])

# 1. Warm-up (Crucial for GPU)
# Run 10 iterations to initialize kernels and memory allocator
for _ in range(10):
    energies, concs, _ = env.sample_batch(CONF['batch_size'])
    activity = physics(energies, concs, receptor_indices)
    loss = loss_fn(activity)
    loss.backward()
    optimizer.step()

# 2. Start Measuring
start_event = torch.cuda.Event(enable_timing=True)
end_event = torch.cuda.Event(enable_timing=True)

start_event.record()

# Run exactly one iteration (or a small fixed number to average)
optimizer.zero_grad()
energies, concs, _ = env.sample_batch(CONF['batch_size'])
activity = physics(energies, concs, receptor_indices)
loss = loss_fn(activity)
loss.backward()
optimizer.step()

end_event.record()

# Wait for the GPU to finish all tasks before calculating time
torch.cuda.synchronize()

print(f"Single iteration time: {start_event.elapsed_time(end_event):.2f} ms")

cuda
tensor([[1, 4, 4, 5, 5],
        [3, 4, 7, 7, 9]])
Single iteration time: 11.56 ms


In [5]:
# 1. SETUP
# -----------------------------------------------------
# We use 1 Unit, 1 Family. The receptor is a homopentamer (Unit 0 five times).

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

# Initialize Modules
conc_strategy = LogNormalConcentration(n_families=CONF['n_families'], init_mean=5.0)
env = LigandEnvironment(CONF['n_units'], CONF['n_families'], conc_model=conc_strategy).to(device)

physics = Receptor(CONF["n_units"], CONF["k_sub"]).to(device)
loss_fn = ExactInformationLoss(k=CONF['k']) # Default bandwidth

# Create the receptor identity: [[0, 0, 0, 0, 0]]
#receptor_indices = torch.zeros(1, CONF["k_sub"], dtype=torch.long, device=device)
receptor_indices = generate_receptor_indices(n_units=CONF['n_units'],k_sub= CONF['k_sub'],n_sensors=CONF['N'])
#receptor_indices = torch.tensor([[0,0,0,0,0],[1,1,1,1,1]],dtype=torch.long)
print(receptor_indices)

# Optimizer
optimizer = optim.Adam(list(env.parameters()) + list(physics.parameters()), lr=CONF["lr"])

# 1. Warm-up (Crucial for GPU)
# Run 10 iterations to initialize kernels and memory allocator
for _ in range(10):
    energies, concs, _ = env.sample_batch(CONF['batch_size'])
    activity = physics(energies, concs, receptor_indices)
    loss = loss_fn(activity)
    loss.backward()
    optimizer.step()

# 2. Start Measuring
start_event = torch.cuda.Event(enable_timing=True)
end_event = torch.cuda.Event(enable_timing=True)

start_event.record()

# Run exactly one iteration (or a small fixed number to average)
optimizer.zero_grad()
energies, concs, _ = env.sample_batch(CONF['batch_size'])
activity = physics(energies, concs, receptor_indices)
loss = loss_fn(activity)
loss.backward()
optimizer.step()

end_event.record()

# Wait for the GPU to finish all tasks before calculating time
torch.cuda.synchronize()

print(f"Single iteration time: {start_event.elapsed_time(end_event):.2f} ms")

cuda
tensor([[0, 2, 3, 4, 5],
        [2, 2, 4, 8, 9]])
Single iteration time: 5.55 ms


In [6]:
from torch.profiler import profile, record_function, ProfilerActivity

# 1. SETUP
# -----------------------------------------------------
# We use 1 Unit, 1 Family. The receptor is a homopentamer (Unit 0 five times).

device = "cuda" if torch.cuda.is_available() else "cpu"
#print(device)

# Initialize Modules
conc_strategy = LogNormalConcentration(n_families=CONF['n_families'], init_mean=5.0)
env = LigandEnvironment(CONF['n_units'], CONF['n_families'], conc_model=conc_strategy).to(device)

physics = Receptor(CONF["n_units"], CONF["k_sub"]).to(device)
loss_fn = ExactInformationLoss(k=CONF['k']) # Default bandwidth

# Create the receptor identity: [[0, 0, 0, 0, 0]]
#receptor_indices = torch.zeros(1, CONF["k_sub"], dtype=torch.long, device=device)
receptor_indices = generate_receptor_indices(n_units=CONF['n_units'],k_sub= CONF['k_sub'],n_sensors=CONF['N'])
#receptor_indices = torch.tensor([[0,0,0,0,0],[1,1,1,1,1]],dtype=torch.long)
#print(receptor_indices)

# Optimizer
optimizer = optim.Adam(list(env.parameters()) + list(physics.parameters()), lr=CONF["lr"])

with profile(
    activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
    record_shapes=True,
    with_stack=True
) as prof:
    with record_function("optimization_step"):
        optimizer.zero_grad()
        
        # We can tag specific blocks to see them in the results
        with record_function("sampling"):
            energies, concs, _ = env.sample_batch(CONF['batch_size'])
            
        with record_function("physics_forward"):
            activity = physics(energies, concs, receptor_indices)
            
        with record_function("loss_computation"):
            loss = loss_fn(activity)
            
        with record_function("backward_pass"):
            loss.backward()
            
        optimizer.step()

# Print the top 10 most expensive GPU kernels
print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=10))

# OPTIONAL: Export to view in Chrome (chrome://tracing)
#prof.export_chrome_trace("trace.json")

-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                      optimization_step         9.38%       1.544ms        82.27%      13.538ms      13.538ms       0.000us         0.00%       1.993ms       1.993ms             1  
                                       loss_computation         2.76%     454.000us        11.97%       1.969ms       1.969ms       0.000us         0.00%       1.219ms       1.219ms             1  
         

STAGE:2026-02-25 15:31:35 20092:20092 ActivityProfilerController.cpp:314] Completed Stage: Warm Up
STAGE:2026-02-25 15:31:35 20092:20092 ActivityProfilerController.cpp:320] Completed Stage: Collection
STAGE:2026-02-25 15:31:35 20092:20092 ActivityProfilerController.cpp:324] Completed Stage: Post Processing


In [7]:
from torch.profiler import profile, record_function, ProfilerActivity

# 1. SETUP
# -----------------------------------------------------
# We use 1 Unit, 1 Family. The receptor is a homopentamer (Unit 0 five times).

device = "cuda" if torch.cuda.is_available() else "cpu"
#print(device)

# Initialize Modules
conc_strategy = LogNormalConcentration(n_families=CONF['n_families'], init_mean=5.0)
env = LigandEnvironment(CONF['n_units'], CONF['n_families'], conc_model=conc_strategy).to(device)

physics = Receptor(CONF["n_units"], CONF["k_sub"]).to(device)
loss_fn = ProxyInformationLoss(cov_weight=CONF['cov_weight']) # Default bandwidth

# Create the receptor identity: [[0, 0, 0, 0, 0]]
#receptor_indices = torch.zeros(1, CONF["k_sub"], dtype=torch.long, device=device)
receptor_indices = generate_receptor_indices(n_units=CONF['n_units'],k_sub= CONF['k_sub'],n_sensors=CONF['N'])
#receptor_indices = torch.tensor([[0,0,0,0,0],[1,1,1,1,1]],dtype=torch.long)
#print(receptor_indices)

# Optimizer
optimizer = optim.Adam(list(env.parameters()) + list(physics.parameters()), lr=CONF["lr"])

with profile(
    activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
    record_shapes=True,
    with_stack=True
) as prof:
    with record_function("optimization_step"):
        optimizer.zero_grad()
        
        # We can tag specific blocks to see them in the results
        with record_function("sampling"):
            energies, concs, _ = env.sample_batch(CONF['batch_size'])
            
        with record_function("physics_forward"):
            activity = physics(energies, concs, receptor_indices)
            
        with record_function("loss_computation"):
            loss = loss_fn(activity)
            
        with record_function("backward_pass"):
            loss.backward()
            
        optimizer.step()

# Print the top 10 most expensive GPU kernels
print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=10))

# OPTIONAL: Export to view in Chrome (chrome://tracing)
#prof.export_chrome_trace("trace.json")

-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                              aten::mul         2.24%     553.000us         3.30%     814.000us      19.854us       2.591ms        31.43%       2.591ms      63.195us            41  
      autograd::engine::evaluate_function: DivBackward0         0.51%     127.000us         3.65%     901.000us     150.167us       0.000us         0.00%       2.568ms     428.000us             6  
         

STAGE:2026-02-25 15:31:42 20092:20092 ActivityProfilerController.cpp:314] Completed Stage: Warm Up
STAGE:2026-02-25 15:31:42 20092:20092 ActivityProfilerController.cpp:320] Completed Stage: Collection
STAGE:2026-02-25 15:31:42 20092:20092 ActivityProfilerController.cpp:324] Completed Stage: Post Processing
